FEUILLE DE TP 08

---
# Intégration numérique (Partie 2) .  Apprentissage du cours <br> (méthodes de Gauss)
---

In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina' 

import numpy as np
from scipy.integrate import quad
import matplotlib.pyplot as plt

Nous avons vu les méthodes de quadratures composées obtenues avec des formules de quadratures élémentaires classiques~: rectangle à gauche, point milieu, trapèze, Simpson. Dans ce TP nous allons étudier sur quelques exemples
1. des méthodes plus avancées permettant d'obtenir un ordre plus élevé avec un même nombre de point : les *méthodes de Gauss-Legendre* et des variantes,

Nous réservons pour une exploration ultérieure,

2. la *méthode de Romberg*, qui utilise un procédé général d'accélération (méthode de Richardson).
3. la méthode *d'adaptation* de pas, qui vise à s'affranchir de la subdivision uniforme dans la méthode composite. 

On utilisera $f_0 : x\mapsto \sin(x+x^2)$ sur $[0,3]$ pour les tests.


## I-  Préliminaires :  Préparation
Recopiez le code de la fonction `quad_composite` définie dans le TP précédent, et de la méthode des trapèzes `trapeze`. Définissez la fonction $f_0$ et les extrémités $a=0, b=3$.

In [ ]:
def f0(x):
    return np.sin(x+x**2)

a = 0
b =3

def quad_composite(f, x, quad_elem):
    print(quad_elem.__name__)
    n = len(x)-1
    I = 0
    for i in range(n):
        I += quad_elem(f, x[i], x[i+1])
    return I

def quad_trapeze(f, a, b):
    """  """
    return (b-a)*(f(a)+f(b))/2

### Question I-1 (  Analyse de d'erreur )
Programmez une fonction `analyse_erreur(f,a,b,meth,title)` qui prend en argument une fonction `f`, les extrémités `a,b` d'un intervalle $[a,b]$, une méthode de quadrature élémentaire `meth` et une chaîne de caractères `title` et qui estime l'ordre de convergence de la méthode composite associée observation de l'évolution de l'erreur $e_h$ en fonction de $h$ obtenue par la méthode composite associée pour des pas $h = 2^{-2}, 2^{-3}, \ldots, 2^{-8}$. 

*On pourra au choix utiliser une approche graphique ou une approche textuelle en s'inspirant de la **Question III-2-1-3** du TP précédent*.

Testez cette fonction sur la méthode des trapèzes. 

## II- Méthodes de Gauss-Lobato (à 2 points)

On a vu que certaines méthodes à $N$ points pouvaient être d'ordre $>N$ (point milieu, Simpson). La méthode à $2$ points que l'on a vue, les trapèzes ($NC_2$), est pourtant d'ordre exactement $2$. Aurait-on pu mieux choisir les points (les extrémités $a$ et $b$ dans le cas des trapèzes) pour obtenir un ordre plus important ? C'est ce que l'on va voir.

### Question II-1 : (théorique)
On considère une formule de quadrature élémentaire à $2$ points $x_1,x_2$ où $x_1 = -1$,
$$\int_{-1}^1 f \approx (\omega_1f(-1) + \omega_2 f(x_2)).$$
On suppose que $\omega_1,\omega_2,x_2$ ont pu être choisi de sorte que la méthode soit d'ordre au moins $3$.
1. En considérant le polynôme $P = (X+1)(X-x_2)$, déterminer $x_2$.
2. En considérant les polynômes $P = 1$ et $P=X$, déterminer $\omega_1,\omega_2$

### Question II-2
On considère la méthode à 2 points $GLob^g_2$
\begin{array}{c|cc}
x_i & -1 & \frac 13\\ \hline
\frac{\omega_i}2& \frac{1}{4} & \frac{3}{4}
\end{array}
Programmez la méthode élémentaire `GLobG2(f,a,b)` associée et la méthode composé correspondante `quad_comp_GLobG2(f,x)`.

On rappelle qu'une méthode élémentaire $\int_{-1}^1 f \approx \sum_{i=1}^N \omega_i f(x_i)$ sur $[-1,1]$ induit la méthode (par changement de variable affine) sur un intervalle $[a,b]$ est
$$\int_a^b f \approx (b-a) \sum_{i=1}^N \frac{\omega_i}2 f\left(\frac{b-a}{2} x_i + \frac{a+b}{2}\right).$$

### Question II-3
Estimez numériquement l'ordre de la méthode. A-t-on obtenu ce que l'on voulait ? 

## III- Méthode de Gauss-Legendre (  à  2 points)

Peut-on faire encore mieux ? En effet, dans la méthode de Gauss-Lobato précédente, on a imposé que $x_1$ soit l'extrémité gauche de l'intervalle, or on aurait pu tout aussi bien imposer que $x_2$ soit l'extrémité droite, et on aurait obtenu la méthode de Gauss-Lobato à droite $GL^d_2$
\begin{array}{c|cc}
x_i & -\frac 13 & 1\\ \hline
\frac{\omega_i}2& \frac{3}{4} & \frac{1}{4}
\end{array}
mais celle-ci est aussi d'ordre $3$. Que se passe-t-il si *aucun* des $x_i$ n'est imposé ?

### Question III-1: (théorique)
On considère une formule de quadrature élémentaire à $2$ points $x_1,x_2$ ,
$$\int_{-1}^1 f \approx (\omega_1f(x_1) + \omega_2 f(x_2)).$$
On suppose que $\omega_1,\omega_2,x_2$ ont pu être choisi de sorte que la méthode soit d'ordre au moins $4$.
1. En considérant les polynômes $P = (X-x_1)(X-x_2)$ et $Q = X(X-x_1)(X-x_2)$, déterminer $x_1,x_2$.
2. En considérant les polynômes $P = 1$ et $P=X$, déterminer $\omega_1,\omega_2$

### Question III-2:
On considère la méthode de Gauss-Legendre à 2 points $GLeg_2$
\begin{array}{c|cc}
x_i & -\frac1{\sqrt 3} & \frac 1{\sqrt 3}\\ \hline
\frac{\omega_i}2& \frac{1}{2} & \frac{1}{2}
\end{array}
Programmez la méthode élémentaire `GLeg2(f,a,b)` associée et la méthode composé correspondante `quad_comp_GLeg2(f,x)`.

### Question III-3:
Estimez numériquement l'ordre de la méthode. A-t-on obtenu ce que l'on voulait ? Comparez à la méthode de Simpson.

## IV- Méthodes de Gauss-Legendre d'ordre supérieur ($\star$)

La méthode de Gauss-Legendre à $N = 2$ points que l'on vient de voir est d'ordre $4 = 2N$. En réalité, il est possible de généraliser cette méthode pour tout $N\geq 1$ : on obtient les méthodes de Gauss-Legendre à $N$ points $GLeg_N$, qui sont toutes d'ordre $2N$. Rappelons que les méthodes de Newton-Cotes à $N$ points ne sont en général que d'ordre $N$, ou $N+1$ lorsque $N$ est impair.

La théorie permet de trouver les points (puis les poids) de ces méthodes comme racines de certains polynômes orthogonaux, ici les **polynômes de Legendre**, dont les racines ne sont pas triviales, ce qui limite l'intérêt de la méthode pour $N$ grand. Pour $N$ petit, les calculs peuvent se faire facilement "à la main".

### Question IV-1 (théorique) 
On considère une formule de quadrature élémentaire à $3$ points $x_1,x_2,x_3$ ,
$$\int_{-1}^1 f \approx (\omega_1f(x_1) + \omega_2 f(x_2)+ \omega_3 f(x_3)).$$
On suppose que la méthode est d'ordre au moins $6$.
1. En supposant que la méthode est symétrique, donnez $x_2$ ainsi qu'une relation entre $x_1$ et $x_3$, et une relation entre $\omega_1$ et $\omega_3$.
1. En considérant le polynômes $P = X^2(X-x_1)(X-x_3)$, déterminer $x_1,x_2,x_3$.
2. En considérant les polynômes $P = 1$ et $P = X^2$, déterminer $\omega_1,\omega_2, \omega_3$.

Résumons ce que nous avons obtenu.

#### Méthodes de Gauss-Legendre pour $N\leq 3$

- $GLeg_1$ (point milieu)
\begin{array}{c|c}
x_i & 0 \\ \hline
\frac{\omega_i}2& 1
\end{array}
- $GLeg_2$
\begin{array}{c|cc}
x_i & -\frac1{\sqrt 3} & \frac 1{\sqrt 3}\\ \hline
\frac{\omega_i}2& \frac{1}{2} & \frac{1}{2}
\end{array}
- $GLeg_3$
\begin{array}{c|ccc}
x_i & -\sqrt{\frac 35} & 0 & \sqrt{\frac 35}\\ \hline
\frac{\omega_i}2& \frac{5}{18} & \frac{8}{18} & \frac{5}{18}
\end{array}

### Question IV-2:
Programmez la méthode de Gauss-Legendre élémentaire à $3$ points `GLeg3(f,a,b)` et la méthode composé correspondante `quad_GLeg3(f,x)`.

### Question IV-3  :
Vérifiez numériquement l'ordre de la méthode $GLeg_3$.

## V- Méthodes de Gauss-Radau ($\star$)

Les méthodes de Gauss-Lobato sont une variante des méthodes de Gauss dans lesquelles on impose qu'une extrémité de l'intervalle fasse partie des points d'interpolation. Une autre variante consiste à imposer que les **$2$ extrémités de l'intervalle** fassent partie des points choisis. Il existe un et un unique choix de points et poids (contenant les extrémités) pour que la méthode obtenue soit d'ordre au moins $2N-2$ : on obtient les *méthodes de Gauss-Radau* à $N$ points $GRad_N$.

### Question  V-1: (théorique)  -- $GRad_3$
On considère une formule de quadrature élémentaire à $3$ points $x_1 = -1,x_2,x_3 = 1$ ,
$$\int_{-1}^1 f \approx (\omega_1f(-1) + \omega_2 f(x_2)+ \omega_3 f(1)).$$
On suppose que la méthode est d'ordre au moins $4$.
1. En considérant le polynômes $P = (X-x_2)(X^2-1)$, déterminer $x_2$.
1. En considérant le polynômes $P = X(X-1)(X+1)$, déterminer $\omega_2$.
2. En considérant les polynômes $P = 1$ et $P = X$, déterminer $\omega_1,\omega_3$.

Qu'obtient-on ? Pouvait-on le prévoir ?


In [3]:
# On peut reprendre les questions IV-2 et IV-3 ci-dessus. 


## VI-  Méthodes de Gauss-Legendre : Vérifications des formules du cours  ($\star$)

*Dans cette partie on va essayer de calculer les points et les poids de la méthode de Gauss-Legendre pour les degrés élevés.* 

Le *polynôme de Legendre* de degré N noté $P_N$ est le polynôme défini par la récurrence:

$$
P_0 (x) = 1, \quad P_1 (x) = x,  \quad   P_N (x) = \frac{(2 N - 1) x P_{N-1}(x) - (N-1) P_{N-2}(x)}{N} \quad \forall x \in [-1, 1] 
$$
La formule de quadrature de Gauss-Legendre à $N$ points sur l'intervalle $[-1,1]$ <br>est alors défine par 
les $N$ racines $x_i$ du polynôme de legendre de degré $N$ associés aux $N$ poids 
$$w_i  = \frac{2}{(1 - x_i^2) P^{'}_N (x_i)^2} = \frac{2(1 - x_i^2)}{N^2 P_{N-1}(x_i)^2}.$$

La difficulté dans cette généralisation est alors le calcul des racines $x_i$ du polynôme de Legendre de degré $N$ quelconque <br> (*Cela se fait en pratique plutôt numériquement*). 
<br> Nous allons dans ce TP plutôt nous servir de 
la fonction `roots_legendre` du module `scipy.special`. <br> 
*Nous verrons dans un exercice d'extension comment les déterminer numériquement.*

**Definitions:**
On remarquera que les extrêmités $-1$ et $1$ n'étant pas les racines du polynôme de Legendre, ils ne feront pas partie des points de quadrature. Ont dit alors que la formule de quadrature de **Gauss-Legendre** est **ouverte**,
alors que celle de **Gauss-Radau** est **fermée** et celles de **Gauss-Lobato** est **semi-ouverte** ou **semi-fermée**.


> Pour les applications de cette partie, on prendra encore la fonction $f_0$ cette fois sur l'intervalle $[-1,1]$.

### Question  VI-1:
> 1. Ecrire une fonction recursive `poly_legendre( x, n )` qui prend en argument un `ndarray` `x` et un entier `n` et qui retourne une évaluation de $n$-ième polynome de Legendre $ P_n $ aux points `x`.
> 2. Tracer les polynomes $ P_n $, $ n = 1, 2, \ldots, 6 $ sur $  [ -1, 1 ]$.
> 3. En se servant du graphique donner des valeurs approchées de points de quadrature $ x_k $, $ k = 0, \ldots, n $ pour $ n = 1, 2, 3 $. Les résultats que vous obtenez pour $ n = 1, 2 $ sont-ils conformes aux résultats vus en cours? 

In [2]:
x = np.linspace( -1, 1, 100 )
...

### Question  VI-2:
> 1. Ecrire une fonction `poids_gauss( x, n )` qui prend en argument un `ndarray` `x` et un entier `n` et qui retourne un `ndarray` des coéficients $ w_1, \ldots, w_n $ de la formule de Gauss à `n` points avec les points de quadrature `x`.
> 2. Testez votre fonction pour $ n = 1 $, $ x = [ 0 ] $ et $ n = 2 $, $ x = [ -\frac{1}{\sqrt{3}}, \frac{1}{ \sqrt{3}} ] $.

La fonction `roots_legendre` de `scipy.special` permet de calculer les valeurs approchées des racines $ x_1, x_2, \ldots, x_n $ du $n$-ième polynome de Legendre ainsi que les poids $ w_1, \ldots, w_n $ de la formule de quadrature de Gauss associée. 

### Question  VI-3:
> 1. En utilisant la fonction `roots_legendre` de `scipy.special` calculer les racines des polynomes de Legendre $ P_n $ pour $ n = 1, \ldots, 6 $. Ajouter une représentation graphique des racines à l'image de la **Question  VI-1**.
> 2. Comparer les coéficients de la formule de quadrature de Gauss à $ n $ points obtenus via la fonction `roots_legendre` et votre fonction `poids_gauss` pour quelques différentes valeurs de $ n $.  

### Question  VI-4:
> 1. Programmez une fonction `quad_gauss( f, x, n )` qui prend en argument une fonction `f` et un entier `n` et qui retourne la valeur approchée de $I$ obtenue par la formule de quadrature de Gauss à $ n $ points.
> 2. Pour $ n = 6 $ vérifier numériquement que la formule de quadratue de Gauss à $ n $ points est exacte pour les polynômes de degré $ 2n -1 $.

### Question  VI-5:
> 1. Comparez la valeur approchée de $I$ obtenue à l'aide de la fonction `quad_gauss` avec sa valeur exacte pour des différents nombres de points $n = 1, 2, \ldots, 13$. Que se passe-t-il quand $n$ augmente ?
> 2. Tracez le logarithme d'erreur entre l'intégrale approchée et la valeur exacte en fonction de $n$. Que observez-vous?

### Question  VI-6:
> Pour les méthodes composées du rectangles, du trapèze et de Simpson tracez sur le meme graphique le logarithme d'erreur entre l'intégrale approchée et la valeur exacte en fonction de $n$, où $ n $ est le nombre de points utilisés pour intégrer la fonction $ f $. Commentez. 